# CS-781 Phase 4 — part 2: the three decoder runs (notebook H2)

The first Phase-4 run finished `prefix_full`, `moe_full`, `encdec_lorma` and `swa_bitfit`, but not the three runs
built on the **decoder** Phase-1 model: `kd_moe_to_decoder4` (distillation — compulsory), `decoder_partial` and
`decoder_lora`. Without them the Phase-4 rules (distillation + all four architectures) are not met.

This notebook is the same pipeline, but it **trains only those three** and **reuses the four finished runs** from the
first run's output: the lab (N11) builds zips, `index.csv` and the score chart from all seven.

**Most likely cause of the failure:** the decoder checkpoint `continued/best_continued.pt` was not attached. In
Phase 3 it came from the input **`notebook27acca3fc3`** (your notebook D, "phase1-continue"). The new **pre-flight
cell (N7b)** checks every input within a minute and stops with a clear message if anything is missing.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · Add-ons → Secrets → **`Capstone_Phase_1` attached**.

**Inputs (Add Input) — all of these:**
| Input | Provides |
|---|---|
| `notebook857159535b` | tokenizer (`bpe_tok.py`, `bpe.json`) |
| **`notebook27acca3fc3`** (notebook D, phase1-continue) | **decoder** checkpoint `continued/best_continued.pt` |
| `phase1-train-extra` (phase1-training2) | **moe** checkpoint — the distillation teacher is rebuilt from it |
| **your first Phase-4 notebook (`phase4-cjpe`) — its output** | the 4 finished runs, the trained teacher `cj_runs/moe_full/best.pt`, the tokenisation cache |

(`Phase1_Training1` and the `prefix` dataset are not needed this time, but attaching them does no harm.)

**How to use:** Save & Run All (~3 h: the three runs, two at a time). If N7b stops the run, it says which input is
missing — add it and run again. Then Output → `submissions/`: `index.csv`, `score_chart.png` and the zips of all seven
runs; download them into your `Phase 4` folder.

## N1. Login + download the Phase 4 data

In [ ]:
import os, shutil, json
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p4data", exist_ok=True)
for path in ["Train/train.json", "Train/dev.json", "Test/test.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_4_CJPE/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p4data/{os.path.basename(path)}")
print({f: f"{os.path.getsize('/kaggle/working/p4data/' + f) / 1e6:.1f} MB" for f in sorted(os.listdir("/kaggle/working/p4data"))})

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments, CJPE options, lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
# =====================================================================================================
#
#  PHASE 4 RULES THIS CONFIG MUST SATISFY (§10)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient);
#    the best four submissions come from DIFFERENT Phase-1 models
#  * ≥ 4 substantially different experiments: ≥ 3 different techniques FROM THE §10.2.1 LIST — Full fine-tuning,
#    Partial/Layer-wise fine-tuning, Layer freezing/unfreezing, BitFit, LORMA, AdaLoRA, Adapters, Knowledge
#    Distillation — LORMA COMPULSORY — and ≥ 1 knowledge distillation.
#    The plan below covers: LoRMA (encdec_lorma), full (prefix_full, moe_full), partial (decoder_partial),
#    BitFit (swa_bitfit), distillation (kd_moe_to_decoder4). LoRA / rsLoRA / IA³ are not on this phase's list.
#  * "handle the long-document nature" — every case is cut into windows; train_chunks + the lab's SELECTS decide
#    which parts of the judgment are used (the brief asks you to compare sections).

CONFIG = {
    "seed": 42,
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",
        "prefix":    "prefix/best.pt",
        "encdec":    "encdec/best.pt",
        "swa":       "swa/best.pt",
        "diffusion": "diffusion/best.pt",
        "moe":       "moe/best.pt",
    },

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        # --- CJPE-specific: which part of a long judgment the model learns from, and how a case gets one label ---
        "train_chunks": "tail",  # windows of each case used in TRAINING (a window = `window` tokens):
                                 #   "tail"     the case's last window      "head"  its first window
                                 #   "headtail" first + last                "all"   every window (at most
                                 #              max_windows_per_case per case per epoch, sampled at random)
                                 # More windows = more training text per case but proportionally more GPU time.
                                 # The lab (N11) compares head / tail / head+tail / all at PREDICTION time anyway.
        "max_windows_per_case": 4,   # only for train_chunks = "all"
        "label_tokens": "all",   # the case's label on every token of a window ("all") or only its last token ("last")
        "pool": "mean",          # how the run's OWN zip reads a window: "mean" of its tokens or its "last" token
        # --- compute: cases are long and many, so each run has a time budget ---
        "time_budget_min": 75,   # YOUR CHOICE. Training stops after this many minutes (or after `epochs`), then the
                                 # best checkpoint is scored on dev + test (~5-15 min more). 7 runs x ~90 min on two
                                 # GPUs ≈ 5.5 h of the 12 h a Kaggle session allows.
        "eval_minutes": 25,      # evaluate on the dev cases every this many minutes (and at each epoch end); the
                                 # best evaluation is the checkpoint that is kept
        "epochs": 3,             # upper limit on passes over the training windows
        # --- the rest as in Phases 2-3 ---
        "window": 1024,          # tokens per window (capped at what the model accepts: encdec 768)
        "context": "plain",
        "lr": 5e-5,              # YOUR CHOICE
        "batch_size": 8,
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "head_dropout": 0.1,
        "class_weights": "none", # "none" or "balanced" (each class weighted by 1 / its share of training cases)
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "train_on_all": False,   # True = train on train + dev (no dev scores)
        "epochs_from": None,     # for train_on_all runs: a finished dev-scored run whose best epoch is reused
        "teacher": None,
        "kd": {"temperature": 2.0, "alpha": 0.5},
        "student_layers": None,
    },

    # ---------------- the experiments — YOUR CHOICE (starting plan = your earlier settings) ----------------
    "experiments": {
        "moe_full":           {"arch": "moe", "method": "full"},                       # also the distillation teacher
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},
        "encdec_lorma":       {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_partial":    {"arch": "decoder", "method": "partial"},  # your method_partial (default n_last=2)
        "swa_bitfit":         {"arch": "swa", "method": "bitfit", "lr": 1e-3},
        "prefix_full":        {"arch": "prefix", "method": "full"},
        "decoder_lora":       {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        # --- optional extras (not in SESSION_EXPERIMENTS below): ---
        "diffusion_full":     {"arch": "diffusion", "method": "full"},
        "decoder_rslora":     {"arch": "decoder", "method": "rslora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_ia3":        {"arch": "decoder", "method": "ia3", "lr": 5e-4},
        # "name_ht":  {<settings of another run>, "train_chunks": "headtail"},     <- section comparison in training
        # "name_all": {<settings of a dev-scored run>, "train_on_all": True, "epochs_from": "<that run>"},
    },
}

# ---------------- which experiments THIS session trains (GPU) and smoke-tests (CPU) ----------------
SESSION_EXPERIMENTS = ["kd_moe_to_decoder4", "decoder_partial", "decoder_lora"]   # part 2: the decoder runs
# (kd first: distillation is compulsory. Its teacher moe_full comes from the first run's output.)          # a distillation student waits for its teacher
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)

# ---------------- lab: post-processing + ensembles of finished runs (N11) — YOUR CHOICE ----------------
SELECTS = ["all", "head", "tail", "headtail", "tail2"]   # parts of the judgment used at prediction time
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]                   # predict 1 when P(1) >= threshold (calibration)
PROB_ENSEMBLES = {}          # {"name": ["run_a", "run_b"]} -> every variant, dev-scored;
                             # {"name": {"members": [...], "select": "tail", "token_agg": "mean", "threshold": 0.5}}
GREEDY = True
RUN_DIR_OVERRIDES = {}
TOP_N_ZIPS = 30

json.dump(CONFIG, open("/kaggle/working/run_config_cj.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
    f = e.get("epochs_from")
    assert not f or (e.get("train_on_all") and f in CONFIG["experiments"]), f"{name}: epochs_from needs train_on_all and a known run"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
n = len(SESSION_EXPERIMENTS)
budget = [CONFIG["experiments"][e].get("time_budget_min", CONFIG["defaults"]["time_budget_min"]) for e in SESSION_EXPERIMENTS]
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {n}: {SESSION_EXPERIMENTS}")
if n:
    print(f"rough GPU time: ~{sum(b + 15 for b in budget) / 2 / 60:.1f} h on T4 x2 (time budgets + scoring)")

## N4. Pipeline code: data, tokenisation cache, windows, window scores, pooling, metrics, backbones, files

In [ ]:
%%writefile cj_common.py
"""Phase 4 (Court Judgment Prediction) pipeline: data, one-time Phase-1 tokenisation of every case (cached on disk),
which windows of a long judgment are used (head / tail / head+tail / all), window-level scores, document pooling,
metrics (same rules as evaluate_phase4.py), Phase-1 backbones that return hidden states, run finder, files."""
import glob, json, os, sys
from multiprocessing import Pool
import numpy as np, torch, torch.nn as nn

INPUT_ROOT = os.environ.get("P4_INPUT", "/kaggle/input")
WORK = os.environ.get("P4_WORK", "/kaggle/working")
DATA = f"{WORK}/p4data"
CACHE = f"{WORK}/p4cache"
SPLITS = ("train", "dev", "test")
IGNORE = -100
N_CLASSES = 2

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

# ------------------------------------------------------------------------------------------------ data
def _label(x):
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, np.integer)) and int(x) in (0, 1):
        return int(x)
    if isinstance(x, str) and x.strip() in ("0", "1"):
        return int(x.strip())
    raise ValueError(f"label must be 0 or 1, got {x!r}")

def read_split(name):
    """-> list of {"id": str, "text": str, "label": 0/1 or None}. Accepts the documented list of
    {"id", "text", "label"} and close variants (a dict keyed by id; "gold" / "labels" instead of "label";
    text given as a list of sentences)."""
    obj = json.load(open(f"{DATA}/{name}.json", encoding="utf-8"))
    items = list(obj.items()) if isinstance(obj, dict) else [(None, x) for x in obj]
    out = []
    for k, it in items:
        if isinstance(it, str):
            it = {"text": it}
        doc_id = it.get("id", it.get("case_id", it.get("doc_id", k)))
        text = it.get("text", it.get("document"))
        if isinstance(text, list):
            text = " ".join(map(str, text))
        if doc_id is None or not isinstance(text, str):
            raise ValueError(f"{name}.json: cannot read an example (keys: {list(it)[:8]}). Expected "
                             '{"id": ..., "text": "...", "label": 0/1}')
        lab = next((it[key] for key in ("label", "gold", "labels", "y") if key in it), None)
        out.append({"id": str(doc_id), "text": text, "label": None if lab is None else _label(lab)})
    return out

# ------------------------------------------------------------------------------------------------ tokenisation cache
_ENC = None
def _init_worker():
    global _ENC
    _ENC = bpe_tok.FastEncoder(bpe_tok_bpe())

def _encode(text):
    return np.asarray(_ENC.encode(text), dtype=np.int32)

def build_cache(workers=None, limit=None):
    """Tokenises every case ONCE with your Phase-1 tokenizer (bpe_tok.FastEncoder, as in pretraining) and stores
    per split: all token ids concatenated (ids.npy) + where each case starts (offsets.npy) + ids/labels (meta.json).
    Training runs then read these files instead of re-tokenising."""
    os.makedirs(CACHE, exist_ok=True)
    for split in SPLITS:
        if os.path.exists(f"{CACHE}/{split}_meta.json"):
            continue
        docs = read_split(split)[:limit]
        with Pool(workers or os.cpu_count(), initializer=_init_worker) as pool:
            arrs = pool.map(_encode, [d["text"] for d in docs], chunksize=16)
        offsets = np.zeros(len(arrs) + 1, dtype=np.int64)
        offsets[1:] = np.cumsum([len(a) for a in arrs])
        dtype = np.uint16 if len(bpe_tok_bpe().itos) <= 65535 else np.int32
        np.save(f"{CACHE}/{split}_ids.npy", np.concatenate(arrs).astype(dtype) if arrs else np.zeros(0, dtype))
        np.save(f"{CACHE}/{split}_offsets.npy", offsets)
        json.dump({"ids": [d["id"] for d in docs], "labels": [d["label"] for d in docs]}, open(f"{CACHE}/{split}_meta.json", "w"))
        print(f"{split}: {len(docs)} cases, {offsets[-1]:,} tokens cached")

class Split:
    """one cached split: ids, labels, and token ids of case i via split.tokens(i) (memory-mapped)"""
    def __init__(self, name):
        m = json.load(open(f"{CACHE}/{name}_meta.json"))
        self.name, self.ids, self.labels = name, m["ids"], m["labels"]
        self.flat = np.load(f"{CACHE}/{name}_ids.npy", mmap_mode="r")
        self.off = np.load(f"{CACHE}/{name}_offsets.npy")
    def __len__(self):
        return len(self.ids)
    def tokens(self, i):
        t = np.asarray(self.flat[self.off[i]:self.off[i + 1]], dtype=np.int32)
        return t if len(t) else np.array([bpe_tok_bpe().stoi["<unk>"]], dtype=np.int32)
    def lengths(self):
        return np.diff(self.off)

# ------------------------------------------------------------------------------------------------ windows of a case
def doc_windows(n, win):
    """non-overlapping windows covering the case; the last one is aligned to the end (so it is the case's last
    `win` tokens). First window = head, last window = tail."""
    if n <= win:
        return [(0, n)]
    starts = list(range(0, n - win, win)) + [n - win]
    return [(s, s + win) for s in sorted(set(starts))]

SELECTS = ("all", "head", "tail", "headtail", "tail2")

def select_windows(wins, how):
    """which windows of a case are used. all | head (first) | tail (last) | headtail (first + last) | tail2 (last two)"""
    if how == "all" or len(wins) == 1:
        return list(range(len(wins)))
    return {"head": [0], "tail": [len(wins) - 1], "headtail": [0, len(wins) - 1],
            "tail2": list(range(max(0, len(wins) - 2), len(wins)))}[how]

def train_windows_for(n, win, how, max_windows, rng):
    """windows a case contributes to training this epoch: head / tail / headtail, or all (a random sample of at
    most max_windows of them when max_windows is set)"""
    wins = doc_windows(n, win)
    if how == "all":
        idx = list(range(len(wins)))
        if max_windows and len(idx) > max_windows:
            idx = sorted(rng.choice(len(idx), max_windows, replace=False).tolist())
    else:
        idx = select_windows(wins, how)
    return [wins[i] for i in idx]

def window_labels(n_tok, label, mode):
    """the case's label on every token of a window (all) or only on its last token (last); others ignored"""
    y = np.full(n_tok, label if mode == "all" else IGNORE, dtype=np.int64)
    y[-1] = label
    return y

def batches(seqs, batch_size, win, shuffle, rng, max_seqs=256):
    """batches of EQUAL-length sequences (your models have no padding mask); short ones packed up to
    about batch_size x window tokens"""
    by_len = {}
    for s in seqs:
        by_len.setdefault(len(s[0]), []).append(s)
    out = []
    for n, group in by_len.items():
        k = max(1, min(max_seqs, (batch_size * win) // n))
        idx = rng.permutation(len(group)) if shuffle else np.arange(len(group))
        for i in range(0, len(group), k):
            out.append([group[j] for j in idx[i:i + k]])
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out

def log_softmax(x):
    x = np.asarray(x, dtype=np.float32)
    m = x.max(-1, keepdims=True)
    return x - m - np.log(np.exp(x - m).sum(-1, keepdims=True))

@torch.no_grad()
def window_scores(model, split, win, device, batch_size=8, max_seqs=256, idx=None):
    """For every case: one row per window (in document order) = [mean over the window's tokens of log p(0),
    log p(1) | log p(0), log p(1) at the window's last token]. -> {case id: (n_windows, 4) float16}"""
    idx = range(len(split)) if idx is None else idx
    jobs = []
    for i in idx:
        t = split.tokens(i)
        for w, (s, e) in enumerate(doc_windows(len(t), win)):
            jobs.append((i, w, t[s:e]))
    n_win = {}
    for i, w, _ in jobs:
        n_win[i] = max(n_win.get(i, 0), w + 1)
    out = {i: np.zeros((n_win[i], 4), dtype=np.float32) for i in n_win}
    by_len = {}
    for j in jobs:
        by_len.setdefault(len(j[2]), []).append(j)
    for L_, group in by_len.items():
        k = max(1, min(max_seqs, (batch_size * win) // L_))
        for a in range(0, len(group), k):
            chunk = group[a:a + k]
            x = torch.tensor(np.stack([c[2] for c in chunk]), dtype=torch.long, device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):
                lp = torch.log_softmax(model(x).float(), -1).cpu().numpy()
            for (i, w, _), l in zip(chunk, lp):
                out[i][w, :2] = l.mean(0)
                out[i][w, 2:] = l[-1]
    return {split.ids[i]: out[i].astype(np.float16) for i in out}

TOKEN_AGGS = ("mean", "last")

def doc_prob(rows, token_agg="mean", select="all"):
    """P(label = 1) of a case from its window rows: take the chosen windows, average their (mean-token or
    last-token) log-probabilities, renormalise."""
    rows = np.asarray(rows, dtype=np.float32)
    sel = select_windows(list(range(len(rows))), select)
    lp = rows[sel][:, :2] if token_agg == "mean" else rows[sel][:, 2:]
    p = np.exp(log_softmax(lp.mean(0)))
    return float(p[1])

def token_aggs_for(label_tokens):
    """a model trained with the label on the last token only has trained scores there"""
    return list(TOKEN_AGGS) if label_tokens == "all" else ["last"]

# ------------------------------------------------------------------------------------------------ metrics
def scores(gold, pred):
    """Macro-F1 over classes 0 and 1 + precision / recall of class 1 (the official secondary metrics) +
    per-class F1 — same rules as evaluate_phase4.py (sklearn, zero_division=0)"""
    g, p = np.asarray(gold), np.asarray(pred)
    f = {}
    for c in (0, 1):
        tp = int(((p == c) & (g == c)).sum()); fp = int(((p == c) & (g != c)).sum()); fn = int(((p != c) & (g == c)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        f[c] = (2 * pr * rc / (pr + rc) if pr + rc else 0.0, pr, rc)
    return {"macro_f1": (f[0][0] + f[1][0]) / 2, "precision": f[1][1], "recall": f[1][2], "f1_0": f[0][0], "f1_1": f[1][0]}

# ------------------------------------------------------------------------------------------------ backbones
TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=1, n_heads=4, max_len=256, max_tgt_len=32, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

class Backbone(nn.Module):
    """Wraps one of YOUR Phase-1 models so that backbone(ids) returns hidden states (B, T, d_model) instead of
    next-token scores (output layer replaced by Identity). encdec: the ENCODER output is used.
    context="echo" (causal models): the window is fed twice and the second copy's states are used (Springer et al. 2024)."""
    def __init__(self, arch, lm, bos_id, context="plain"):
        super().__init__()
        self.arch, self.lm, self.bos_id, self.context = arch, lm, bos_id, context
        self.kind = getattr(lm, "kind", "causal")
        self.d_model = lm.tok_emb.embedding_dim
        self.max_input = lm.max_len // 2 if context == "echo" else lm.max_len
        lm.head = nn.Identity()
        if arch == "encdec":
            del lm.dec_layers, lm.dec_norm, lm.tgt_pos_emb

    def forward(self, ids):
        T0 = ids.size(1)
        x = torch.cat([ids, ids], 1) if self.context == "echo" else ids
        T = x.size(1)
        if self.arch == "encdec":
            lm = self.lm
            h = lm.tok_emb(x) + lm.src_pos_emb(torch.arange(T, device=x.device))
            for layer in lm.enc_layers:
                h = layer(h)
            h = lm.enc_norm(h)
        elif self.kind == "prefix":
            h = self.lm(x, prefix_len=T)
        else:
            h = self.lm(x)
        return h[:, T - T0:] if self.context == "echo" else h

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

def load_backbone(arch, ckpt_pattern, context="plain", smoke=False):
    bpe = bpe_tok_bpe()
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return Backbone(arch, lm, bpe.stoi["<s>"], context)

def iter_adaptable(module, skip=("head",)):
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    return total, sum(p.numel() for p in model.parameters() if p.requires_grad)

# ------------------------------------------------------------------------------------------------ runs + files
def find_runs(filename, overrides=None):
    """{experiment: folder} for every '.../cj_runs/<experiment>/<filename>' under /kaggle/working and /kaggle/input.
    Found more than once: /kaggle/working wins, then the newest file."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{filename}", recursive=True):
            folder = os.path.dirname(p)
            if os.path.basename(os.path.dirname(folder)) != "cj_runs":
                continue
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    out = {e: f for e, (_, f) in sorted(hits.items())}
    out.update(overrides or {})
    return out

def write_submission(ids, preds, path_json, path_zip=None):
    """official format: [{"id": "...", "pred": 0 or 1}, ...], one entry per test case, in test order"""
    json.dump([{"id": i, "pred": int(preds[i])} for i in ids], open(path_json, "w", encoding="utf-8"))
    if path_zip:
        import zipfile
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `cj_methods.py` — YOUR CODE (unchanged from Phase 3; only its import now reads `cj_common`)
Works as-is for CJPE: the head gets 2 outputs. Nothing to write this phase — the §10.2.1 list (full, partial,
BitFit, LoRMA, distillation, …) is covered by methods you already have.

In [ ]:
%%writefile cj_methods.py
# =====================================================================================================
#  CS-781 Phase 4 (copied from Phase 3) — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import cj_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}


# =====================================================================================================
#  6. YOUR PHASE-3 ADDITIONS (rsLoRA, IA³) — unchanged. Note for Phase 4: the §10.2.1 list is Full fine-tuning,
#     Partial/Layer-wise fine-tuning, Layer freezing/unfreezing, BitFit, LORMA, AdaLoRA, Adapters, Knowledge
#     Distillation. rsLoRA, IA³ and LoRA are NOT on it, so they are optional extras this phase. You already have
#     full, partial (method_partial above), BitFit, LoRMA and distillation — no new method is needed.
# =====================================================================================================

def method_rslora(model, r=8, alpha=16, skip=("head",), **cfg):
    """Rank-stabilised LoRA: use alpha / sqrt(r) as the update scale."""
    if r <= 0:
        raise ValueError("r must be a positive integer")

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out, d_in = W.shape
        adapter = LoRAParam(d_out, d_in, r, alpha)
        adapter.scale = alpha / math.sqrt(r)
        parametrize.register_parametrization(owner, pname, adapter)


# -----------------------------------------------------------------------------------------------------
#  B. IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations" (Liu et al. 2022)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    IA³ adds no matrices at all. Every adapted weight gets ONE learned vector, with one number per output
#    row of the weight. Each row of the weight is multiplied by its number. The vector starts as all ones, so
#    at the start the model is unchanged (the same idea as B starting at zero in LoRA). Only these vectors and
#    the head are trained, so IA³ trains far fewer parameters than LoRA.
#
#  What you have to write — two pieces:
#
#    Piece 1 — a small class, written just above method_ia3, shaped like your LoRAParam:
#      1. It inherits from nn.Module and its __init__ receives d_out (the number of output rows of the weight).
#      2. In __init__, after super().__init__(), create one trainable vector of length d_out filled with ones
#         (an nn.Parameter wrapped around torch.ones of size d_out). Call it something like "l".
#      3. Its forward receives the original weight W (shape d_out x d_in) and returns W with every row multiplied
#         by the matching number of the vector. To do that, turn the vector into a column of shape d_out x 1
#         (index it with [:, None]) and multiply it with W — PyTorch repeats the column across all d_in columns.
#
#    Piece 2 — the function method_ia3:
#      4. Go through every adaptable weight with the same loop as method_lora.
#      5. Read the weight's number of output rows (the first number of W.shape).
#      6. Create your Piece-1 object with that number and attach it with parametrize.register_parametrization,
#         exactly like method_lora attaches LoRAParam.
#      7. Delete the "raise NotImplementedError" line.
#
#    For your report: the paper rescales only keys, values and the feed-forward hidden layer. Rescaling every
#    adaptable weight (what steps 4-6 do) is a simpler variant — say which one you used.

class IA3Param(nn.Module):
    """Row-wise IA³ scaling for a weight matrix."""

    def __init__(self, d_out):
        super().__init__()
        self.l = nn.Parameter(torch.ones(d_out))

    def forward(self, W):
        return self.l[:, None] * W


def method_ia3(model, skip=("head",), **cfg):
    """Apply trainable row-wise scaling to every adaptable backbone weight."""
    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out = W.shape[0]
        parametrize.register_parametrization(
            owner,
            pname,
            IA3Param(d_out),
        )


# Do not change this line: it makes "rslora" and "ia3" usable as "method" in N3.
METHODS.update({"rslora": method_rslora, "ia3": method_ia3})

## N7. Scripts: training (one experiment), the lab, the official Phase-4 evaluator

In [ ]:
%%writefile train_cj.py
"""Runs ONE experiment from run_config_cj.json: your Phase-1 backbone + your head (TokenClassifier with 2 outputs)
+ your fine-tuning method. Trains on windows of the training cases (train_chunks), evaluates on the dev cases every
eval_minutes and at the end, keeps the best checkpoint (dev macro-F1), stops at time_budget_min or after `epochs`.
Writes the window scores of every dev / test case (dev_scores.pkl, test_scores.pkl) for the lab (N11), plus its own
dev / test predictions and Codabench zip.
   python train_cj.py --exp NAME --device cuda:0        (real run)
   python train_cj.py --exp NAME --device cpu --smoke   (tiny random model, few cases: checks the code)"""
import argparse, json, os, pickle, random, time
import numpy as np, torch, torch.nn.functional as F
import cj_common as L
import cj_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
cuda = dev.startswith("cuda")
if cuda:
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_cj.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'cj_runs_smoke' if args.smoke else 'cj_runs'}"

def best_epoch_of(name):
    p = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(p):
        p = f"{L.find_runs('summary.json')[name]}/summary.json"
    return json.load(open(p))["best_epoch"]

if E["train_on_all"] and E.get("epochs_from") and not args.smoke:
    E["epochs"] = best_epoch_of(E["epochs_from"])         # may be fractional (an epoch is cut by the time budget)
if args.smoke:
    E.update(window=96, epochs=1, batch_size=4, time_budget_min=3, eval_minutes=99)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ data (tokenised once in N8)
train, dsplit, test = L.Split("train"), L.Split("dev"), L.Split("test")
train_idx = list(range(len(train)))
dev_idx = list(range(len(dsplit)))
test_idx = list(range(len(test)))
if args.smoke:
    train_idx, dev_idx, test_idx = train_idx[:8], dev_idx[:4], test_idx[:4]
dev_gold = {dsplit.ids[i]: dsplit.labels[i] for i in dev_idx}   # dev.json repeats some ids: each id counted once
dev_ids = list(dev_gold)                                          # (its last occurrence, as in the saved scores)
sources = [(train, i) for i in train_idx] + ([(dsplit, i) for i in dev_idx] if E["train_on_all"] else [])

# ------------------------------------------------------------------ model
def build(e, smoke):
    backbone = L.load_backbone(e["arch"], C["checkpoints"][e["arch"]], context=e["context"], smoke=smoke)
    if e.get("student_layers"):
        backbone = M.make_student(backbone, e["student_layers"])
    model = M.TokenClassifier(backbone, n_tags=L.N_CLASSES, dropout=e["head_dropout"])
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    for p in model.head.parameters():
        p.requires_grad_(True)
    return model

model = build(E, args.smoke).to(dev)
win = min(E["window"], model.backbone.max_input)
teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = f"{L.find_runs('best.pt')[E['teacher']]}/best.pt"
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    win = min(win, teacher.backbone.max_input)
total, trainable = L.count_params(model)
labels_all = [s.labels[i] for s, i in sources]
say(event="start", arch=E["arch"], method=E["method"], params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), window=win, train_chunks=E["train_chunks"],
    label_tokens=E["label_tokens"], train_cases=len(sources), label_1_share=round(float(np.mean(labels_all)), 3),
    dev_cases=len(dev_idx), time_budget_min=E["time_budget_min"], teacher=E.get("teacher"))

# ------------------------------------------------------------------ training setup
weights = None
if E["class_weights"] == "balanced":                 # weight of class c ∝ 1 / (its share of training cases)
    share = np.bincount(labels_all, minlength=2) / len(labels_all)
    w = 1.0 / np.maximum(share, 1e-6); w = w / w.mean()
    weights = torch.tensor(w, dtype=torch.float32, device=dev)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
rng = np.random.RandomState(seed)

def epoch_windows():
    out = []
    for s, i in sources:
        t = s.tokens(i)
        for a, b in L.train_windows_for(len(t), win, E["train_chunks"], E["max_windows_per_case"], rng):
            out.append((t[a:b], s.labels[i]))           # per-token labels are built per batch (memory)
    return out

steps_per_epoch = len(L.batches(epoch_windows(), E["batch_size"], win, False, rng))
budget_s = E["time_budget_min"] * 60
total_steps = int(steps_per_epoch * E["epochs"]) if E["train_on_all"] else steps_per_epoch * int(np.ceil(E["epochs"]))
warm = max(1, int(E["warmup_frac"] * min(total_steps, steps_per_epoch)))
def lr_at(step):                                 # warm-up, then linear decay over the planned steps (floor 10%)
    return E["lr"] * min((step + 1) / warm, max(0.1, (total_steps - step) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)
select = E["train_chunks"] if E["train_chunks"] in L.SELECTS else "all"

def dev_eval():
    model.eval()
    sc = L.window_scores(model, dsplit, win, dev, E["batch_size"], idx=dev_idx)
    model.train()
    pred = {i: int(L.doc_prob(sc[i], E["pool"], select) >= 0.5) for i in dev_ids}
    return sc, L.scores([dev_gold[i] for i in dev_ids], [pred[i] for i in dev_ids])

# ------------------------------------------------------------------ train
best, best_epoch, step, last_eval, stop = -1.0, 0.0, 0, time.time(), False
model.train()
if cuda:
    torch.cuda.reset_peak_memory_stats()
epoch = 0
while not stop:
    epoch += 1
    losses = []
    for bi, batch in enumerate(L.batches(epoch_windows(), E["batch_size"], win, True, rng)):
        if (args.smoke and bi >= 6) or step >= total_steps or time.time() - t0 > budget_s:
            stop = True
            break
        x = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.long, device=dev)
        y = torch.tensor(np.stack([L.window_labels(len(b[0]), b[1], E["label_tokens"]) for b in batch]),
                         dtype=torch.long, device=dev)
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            logits = model(x).float()
            if teacher is not None:
                with torch.no_grad():
                    t_logits = teacher(x).float()
                loss = M.kd_loss(logits, t_logits, y, **E["kd"])
            else:
                loss = F.cross_entropy(logits.reshape(-1, 2), y.reshape(-1), weight=weights, ignore_index=L.IGNORE)
            aux = getattr(model.backbone.lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
        if not E["train_on_all"] and time.time() - last_eval > E["eval_minutes"] * 60:
            _, s = dev_eval(); last_eval = time.time()
            ep = round(step / steps_per_epoch, 3)
            say(event="eval", epoch=ep, step=step, train_loss=round(float(np.mean(losses[-200:])), 4),
                dev_macro_f1=round(s["macro_f1"], 4), dev_f1_0=round(s["f1_0"], 4), dev_f1_1=round(s["f1_1"], 4))
            if s["macro_f1"] > best:
                best, best_epoch = s["macro_f1"], ep
                torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")
    if step >= total_steps:
        stop = True
    ep = round(step / steps_per_epoch, 3)
    if losses:
        say(event="epoch_end" if not stop else "train_end", epoch=ep, step=step, train_loss=round(float(np.mean(losses)), 4))
    if E["train_on_all"]:
        torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt"); best_epoch = ep
    else:
        _, s = dev_eval(); last_eval = time.time()
        say(event="eval", epoch=ep, step=step, dev_macro_f1=round(s["macro_f1"], 4),
            dev_f1_0=round(s["f1_0"], 4), dev_f1_1=round(s["f1_1"], 4))
        if s["macro_f1"] > best:
            best, best_epoch = s["macro_f1"], ep
            torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")

# ------------------------------------------------------------------ final scores with the best checkpoint
model.load_state_dict(torch.load(f"{OUT}/best.pt", map_location=dev, weights_only=False)["state"])
model.eval()
peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "train_chunks": E["train_chunks"],
           "label_tokens": E["label_tokens"], "pool": E["pool"], "seed": seed, "train_on_all": E["train_on_all"],
           "best_epoch": best_epoch, "epochs_done": round(step / steps_per_epoch, 3), "steps": step,
           "params_total": total, "params_trainable": trainable, "peak_gpu_gb": peak, "window": win,
           "train_minutes": round((time.time() - t0) / 60, 1)}
if not E["train_on_all"]:
    dsc = L.window_scores(model, dsplit, win, dev, E["batch_size"], idx=dev_idx)
    pickle.dump(dsc, open(f"{OUT}/dev_scores.pkl", "wb"))
    dpred = {i: int(L.doc_prob(dsc[i], E["pool"], select) >= 0.5) for i in dev_ids}
    s = L.scores([dev_gold[i] for i in dev_ids], [dpred[i] for i in dev_ids])
    summary.update(dev_macro_f1=round(s["macro_f1"], 4), dev_precision=round(s["precision"], 4),
                   dev_recall=round(s["recall"], 4), dev_f1_0=round(s["f1_0"], 4), dev_f1_1=round(s["f1_1"], 4))
ti = time.time()
tsc = L.window_scores(model, test, win, dev, E["batch_size"], idx=test_idx)
pickle.dump(tsc, open(f"{OUT}/test_scores.pkl", "wb"))
summary["test_predict_sec"] = round(time.time() - ti, 1)
tids = [test.ids[i] for i in test_idx]
L.write_submission(tids, {i: int(L.doc_prob(tsc[i], E["pool"], select) >= 0.5) for i in tids},
                   f"{OUT}/prediction.json", f"{OUT}/submission_{args.exp}.zip")
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **summary)

In [ ]:
%%writefile cj_lab.py
"""Phase 4 lab — no training. From the window scores every finished run saved (dev_scores.pkl, test_scores.pkl) it
builds prediction sets: which part of the judgment is used (SELECTS: all windows / head / tail / head+tail / last
two) x how a window is read (TOKEN_AGGS: mean of its tokens / its last token) x decision threshold on P(label=1);
probability ensembles; greedy ensembles. Scores each on the official dev cases (macro-F1), writes Codabench zips,
submissions/index.csv and a score chart. You choose which zips to upload."""
import csv, glob, hashlib, json, os, pickle
import numpy as np
import cj_common as L

KIND_COLORS = {"single": "#2a78d6", "ensemble": "#eb6834", "greedy": "#1baf7a"}   # fixed categorical order


class Lab:
    def __init__(self, out_dir=f"{L.WORK}/submissions"):
        self.dev, self.test = L.Split("dev"), L.Split("test")
        # dev.json repeats some case ids: each id is scored ONCE (its last occurrence), like the official evaluator
        self.gold = dict(zip(self.dev.ids, self.dev.labels))
        self.dev_ids = list(self.gold)
        self.out_dir = out_dir
        os.makedirs(f"{out_dir}/dev_predictions", exist_ok=True)
        json.dump([{"id": i, "gold": g} for i, g in self.gold.items()], open(f"{out_dir}/dev_predictions/dev_truth.json", "w"))
        self.rows, self.models, self.cache = [], {}, {}

    def load_runs(self, overrides=None):
        for e, f in L.find_runs("test_scores.pkl", overrides).items():
            s = json.load(open(f"{f}/summary.json")) if os.path.exists(f"{f}/summary.json") else {}
            dv = f"{f}/dev_scores.pkl"
            self.models[e] = {"test": pickle.load(open(f"{f}/test_scores.pkl", "rb")),
                              "dev": pickle.load(open(dv, "rb")) if os.path.exists(dv) and not s.get("train_on_all") else None,
                              "aggs": L.token_aggs_for(s.get("label_tokens", "all")), "folder": f}
        print(f"runs found: {len(self.models)}")
        for e, m in self.models.items():
            print(f"  {e:28s} token aggs {m['aggs']} {'' if m['dev'] is not None else '(no dev scores: train_on_all)'} <- {m['folder']}")

    def probs(self, members, split, agg, select):
        """P(1) per case, averaged over members (probability ensemble)"""
        key = (tuple(members), split, agg, select)
        if key not in self.cache:
            ids = self.dev_ids if split == "dev" else self.test.ids
            self.cache[key] = {i: float(np.mean([L.doc_prob(self.models[m][split][i], agg, select) for m in members]))
                               for i in ids}
        return self.cache[key]

    def preds(self, members, split, agg, select, thr):
        if split == "dev" and any(self.models[m]["dev"] is None for m in members):
            return None
        return {i: int(p >= thr) for i, p in self.probs(members, split, agg, select).items()}

    def add(self, name, kind, dv, tp, **info):
        s = L.scores([self.gold[i] for i in self.dev_ids], [dv[i] for i in self.dev_ids]) if dv is not None else None
        r = lambda k: None if s is None else round(s[k], 4)
        row = {"name": name, "kind": kind, "dev_f1": r("macro_f1"), "dev_p": r("precision"), "dev_r": r("recall"),
               "dev_f1_0": r("f1_0"), "dev_f1_1": r("f1_1"),
               "pred_1_share": round(float(np.mean(list(tp.values()))), 3), **info}
        self.rows.append((row, dv, tp))
        return row["dev_f1"]

    def aggs_of(self, members):
        return [a for a in L.TOKEN_AGGS if all(a in self.models[m]["aggs"] for m in members)]

    def variants(self, name, kind, members, selects, thresholds):
        for agg in self.aggs_of(members):
            for sel in selects:
                for thr in thresholds:
                    self.add(f"{name}__{sel}__{agg}__thr{thr}", kind, self.preds(members, "dev", agg, sel, thr),
                             self.preds(members, "test", agg, sel, thr), members="+".join(members),
                             select=sel, token_agg=agg, threshold=thr)

    def singles(self, selects, thresholds):
        for e, m in self.models.items():
            if m["dev"] is None:
                print(f"{e}: no dev scores (train_on_all) -> its zip is the one train_cj.py wrote")
                continue
            self.variants(e, "single", [e], selects, thresholds)

    def ensembles(self, groups, selects, thresholds):
        """{name: [members]} -> every variant, dev-scored; {name: {"members", "select", "token_agg", "threshold"}}
        -> that one variant (for train_on_all runs, which have no dev scores)"""
        for name, g in groups.items():
            members = g["members"] if isinstance(g, dict) else list(g)
            missing = [m for m in members if m not in self.models]
            if missing:
                print(f"ensemble {name}: skipped, no scores for {missing}")
                continue
            if not isinstance(g, dict):
                self.variants(f"ens_{name}", "ensemble", members, selects, thresholds)
                continue
            a, sel, thr = g["token_agg"], g["select"], g["threshold"]
            self.add(f"ens_{name}__{sel}__{a}__thr{thr}", "ensemble", self.preds(members, "dev", a, sel, thr),
                     self.preds(members, "test", a, sel, thr), members="+".join(members), select=sel, token_agg=a, threshold=thr)

    def greedy(self, selects, thresholds):
        """Greedy forward ensemble selection on dev (Caruana et al. 2004), one per token aggregation: repeatedly
        add the run that raises dev macro-F1 the most (best select x threshold), stop when nothing helps."""
        for agg in L.TOKEN_AGGS:
            pool = [e for e, m in self.models.items() if m["dev"] is not None and agg in m["aggs"]]
            def objective(members):
                best = (-1.0, None, None)
                for sel in selects:
                    for thr in thresholds:
                        p = self.preds(members, "dev", agg, sel, thr)
                        f = L.scores([self.gold[i] for i in self.dev_ids], [p[i] for i in self.dev_ids])["macro_f1"]
                        if f > best[0]:
                            best = (f, sel, thr)
                return best
            chosen, best = [], (-1.0, None, None)
            while True:
                tries = [(objective(chosen + [e]), e) for e in pool if e not in chosen]
                if not tries:
                    break
                res, e = max(tries, key=lambda x: x[0][0])
                if res[0] <= best[0]:
                    break
                chosen.append(e); best = res
            if len(chosen) < 2:
                print(f"greedy {agg}: no ensemble beats the best single run ({chosen})")
                continue
            f, sel, thr = best
            self.add(f"greedy__{sel}__{agg}__thr{thr}", "greedy", self.preds(chosen, "dev", agg, sel, thr),
                     self.preds(chosen, "test", agg, sel, thr), members="+".join(chosen), select=sel, token_agg=agg, threshold=thr)
            print(f"greedy {agg}: {chosen} -> dev macro-F1 {f:.4f} ({sel}, threshold {thr})")

    def finish(self, top_n=30):
        """zips for: the top_n sets by dev macro-F1, the best set of each run / ensemble, every set without dev
        scores. Identical test predictions are written once. index.csv lists every set."""
        scored = sorted([r for r in self.rows if r[0]["dev_f1"] is not None], key=lambda r: -r[0]["dev_f1"])
        keep = {id(r) for r in scored[:top_n]}
        best_of = {}
        for r in scored:
            best_of.setdefault(r[0].get("members"), r)
        keep |= {id(r) for r in best_of.values()} | {id(r) for r in self.rows if r[0]["dev_f1"] is None}
        order = scored + [r for r in self.rows if r[0]["dev_f1"] is None]
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/dev_predictions/*__*.json"):
            os.remove(f)
        seen, n = {}, 0
        for r in order:
            row, dv, tp = r
            row["file"] = ""
            if id(r) not in keep:
                continue
            h = hashlib.md5(json.dumps([tp[i] for i in self.test.ids]).encode()).hexdigest()
            if h in seen:
                row["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = row["file"] = f"{row['name']}.zip"
            L.write_submission(self.test.ids, tp, f"{self.out_dir}/_prediction.json", f"{self.out_dir}/{row['file']}")
            if dv is not None:
                json.dump([{"id": i, "pred": dv[i]} for i in self.dev_ids], open(f"{self.out_dir}/dev_predictions/{row['name']}.json", "w"))
            n += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "kind", "dev_f1", "dev_p", "dev_r", "dev_f1_0", "dev_f1_1", "pred_1_share", "members",
                "select", "token_agg", "threshold", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for row, _, _ in order:
                w.writerow(row)
        print(f"{len(self.rows)} prediction sets scored, {n} zips written to {self.out_dir}/ (index.csv lists all)")
        return [r[0] for r in order]


def table(rows, n=25):
    scored = [r for r in rows if r["dev_f1"] is not None]
    print(f"{'dev F1':>7s} {'P':>6s} {'R':>6s} {'F1 c0':>6s} {'F1 c1':>6s} {'pred1':>6s}  {'prediction set':58s} zip")
    for r in scored[:n]:
        print(f"{r['dev_f1']:7.4f} {r['dev_p']:6.3f} {r['dev_r']:6.3f} {r['dev_f1_0']:6.3f} {r['dev_f1_1']:6.3f} "
              f"{r['pred_1_share']:6.3f}  {r['name'][:58]:58s} {r.get('file', '')}")
    for r in rows:
        if r["dev_f1"] is None and r.get("file"):
            print(f"{'n/a':>7s} {'':6s} {'':6s} {'':6s} {'':6s} {r['pred_1_share']:6.3f}  {r['name'][:58]:58s} {r['file']}")


def chart(rows, path, n=25):
    """Tentative score chart (dev macro-F1, the official metric on the official dev cases): left = the n best
    prediction sets coloured by kind; right = the best set of every run / ensemble with the F1 of each class.
    index.csv is the table view of the same numbers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    scored = [r for r in rows if r["dev_f1"] is not None]
    if not scored:
        print("no dev scores to plot"); return
    top = scored[:n][::-1]
    best_of = {}
    for r in scored:
        best_of.setdefault(r["members"], r)
    groups = list(best_of.values())[:15][::-1]
    ink, muted, grid = "#0b0b0b", "#52514e", "#e4e3df"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, max(6, 0.32 * max(len(top), 3 * len(groups) / 1.6))),
                                 gridspec_kw={"width_ratios": [1.15, 1]})
    fig.patch.set_facecolor("#fcfcfb")
    for ax in (a1, a2):
        ax.set_facecolor("#fcfcfb")
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(grid)
        ax.tick_params(colors=muted, labelsize=8, length=0)
        ax.grid(axis="x", color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
    y = np.arange(len(top))
    a1.barh(y, [r["dev_f1"] for r in top], height=0.62, color=[KIND_COLORS[r["kind"]] for r in top],
            edgecolor="#fcfcfb", linewidth=2)
    a1.set_yticks(y, [r["name"][:58] for r in top], fontsize=7.5, color=ink)
    for yi, r in zip(y, top):
        a1.text(r["dev_f1"] + 0.003, yi, f"{r['dev_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a1.set_xlim(0, min(1.0, max(r["dev_f1"] for r in top) * 1.12))
    a1.set_xlabel("dev macro-F1 (classes 0 and 1)", color=muted, fontsize=9)
    a1.set_title(f"Top {len(top)} prediction sets", loc="left", color=ink, fontsize=11)
    names = {"single": "single run", "ensemble": "your ensemble", "greedy": "greedy ensemble"}
    kinds = [k for k in KIND_COLORS if any(r["kind"] == k for r in top)]
    a1.legend([plt.Rectangle((0, 0), 1, 1, color=KIND_COLORS[k]) for k in kinds], [names[k] for k in kinds],
              loc="lower right", frameon=False, fontsize=8)
    series = [("dev_f1", "macro-F1", "#2a78d6"), ("dev_f1_0", "F1 of class 0", "#eb6834"), ("dev_f1_1", "F1 of class 1", "#1baf7a")]
    yy = np.arange(len(groups)); h = 0.26
    for k, (key, lab, col) in enumerate(series):
        a2.barh(yy + (1 - k) * h, [r[key] or 0 for r in groups], height=h, color=col, label=lab, edgecolor="#fcfcfb", linewidth=1.5)
    for yi, r in zip(yy, groups):
        a2.text(r["dev_f1"] + 0.003, yi + h, f"{r['dev_f1']:.3f}", va="center", fontsize=7.5, color=muted)
    a2.set_yticks(yy, [r["members"][:48] for r in groups], fontsize=7.5, color=ink)
    a2.set_xlim(0, min(1.0, max(max(r[k] or 0 for r in groups) for k, _, _ in series) * 1.12))
    a2.set_xlabel("dev scores of the best set of each run / ensemble", color=muted, fontsize=9)
    a2.set_title("Best per run, per class", loc="left", color=ink, fontsize=11)
    a2.legend(loc="lower right", frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("chart saved:", path)

In [ ]:
%%writefile evaluate_phase4.py
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

## N7b. Pre-flight check — every input present? (stops here if not)

In [ ]:
# N7b — PRE-FLIGHT: checks every input this session needs and STOPS here (within a minute, not after hours) if one
# is missing. Read the printout: every line must show a path, not MISSING.
import json, glob, os
import cj_common as L
C = json.load(open("/kaggle/working/run_config_cj.json"))
problems = []
print("tokenizer:", L.find_input("bpe_tok.py"), "|", L.find_input("bpe.json"))
archs = set()
for e in SESSION_EXPERIMENTS:
    x = C["experiments"][e]
    archs.add(x["arch"])
    if x.get("teacher"):
        archs.add(C["experiments"][x["teacher"]]["arch"])        # the teacher is rebuilt from its Phase-1 model too
for arch in sorted(archs):
    pat = C["checkpoints"][arch]
    hits = sorted(glob.glob(f"{L.INPUT_ROOT}/**/{pat}", recursive=True))
    print(f"Phase-1 checkpoint {arch:8s} ({pat}): {hits[0] if hits else 'MISSING'}")
    if not hits:
        problems.append(f"the Phase-1 '{arch}' checkpoint '{pat}' is not in any input — add the notebook / dataset "
                        f"that contains it" + (" (decoder: notebook27acca3fc3 = notebook D)" if arch == "decoder" else
                                               " (moe: phase1-train-extra / phase1-training2)" if arch == "moe" else ""))
for e in SESSION_EXPERIMENTS:
    t = C["experiments"][e].get("teacher")
    if t and t not in SESSION_EXPERIMENTS:
        f = L.find_runs("best.pt").get(t)
        print(f"distillation teacher '{t}' for {e}: {f + '/best.pt' if f else 'MISSING'}")
        if not f:
            problems.append(f"the trained teacher '{t}' (cj_runs/{t}/best.pt) was not found — add the OUTPUT of your "
                            f"first Phase-4 notebook as input")
earlier = L.find_runs("test_scores.pkl")
print("finished runs found in the inputs (they join the lab):", sorted(earlier) or "none")
missing_earlier = [e for e in ("prefix_full", "moe_full", "encdec_lorma", "swa_bitfit") if e not in earlier]
if missing_earlier:
    print("WARNING: not found:", missing_earlier, "— add the first Phase-4 notebook's output as input, or they are "
          "missing from the lab (not fatal)")
if problems:
    raise RuntimeError("PRE-FLIGHT FAILED — nothing was trained:\n  - " + "\n  - ".join(problems))
print("\nPRE-FLIGHT OK")

## N8. Data check + one-time tokenisation (sizes, label balance, case lengths, checkpoints)

In [ ]:
# N8 — read the data, tokenise every case ONCE (all CPU cores; cached in /kaggle/working/p4cache), then show sizes,
# label balance and case lengths. If this cell fails, the data format differs from the brief: paste the error.
import json, time, importlib, numpy as np, torch
import cj_common as L
importlib.reload(L)
import glob, os, shutil
prev = sorted(glob.glob(f"{L.INPUT_ROOT}/**/p4cache/test_meta.json", recursive=True))
if prev and not os.path.exists(L.CACHE):                 # the first run's cache: no need to tokenise again
    shutil.copytree(os.path.dirname(prev[0]), L.CACHE)
    print("reusing the tokenisation cache from", os.path.dirname(prev[0]))
t0 = time.time()
L.build_cache()
print(f"tokenised in {time.time() - t0:.0f}s")
C = json.load(open("/kaggle/working/run_config_cj.json"))
win = C["defaults"]["window"]
for name in L.SPLITS:
    s = L.Split(name)
    n = s.lengths()
    lab = [x for x in s.labels if x is not None]
    share = f"label 1: {np.mean(lab):.3f} of {len(lab)}" if lab else "no labels"
    print(f"{name:5s}: {len(s):6d} cases | tokens per case median {int(np.median(n))}, 90% {int(np.percentile(n, 90))}, "
          f"max {n.max()} | windows of {win}: median {int(np.median(np.ceil(n / win)))} | {share}")
test = L.Split("test")
assert len(set(test.ids)) == len(test.ids), "test case ids repeat"
dv = L.Split("dev")
print(f"dev: {len(dv.ids) - len(set(dv.ids))} repeated case ids (each id is scored once, like the official evaluator)")
print("example test id:", test.ids[0])
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        path = L.find_input(C["checkpoints"][arch])
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print(f"checkpoint {arch:9s}: {path} | val_loss {ck.get('val_loss', float('nan')):.3f}")
    except Exception as e:
        print(f"checkpoint {arch:9s}: MISSING -> {e}")

## N9. CPU smoke test of the session's experiments

In [ ]:
# N9 — CPU smoke test (tiny random models, a few cases): every experiment must end with exit code 0.
import json, subprocess, sys
C = json.load(open("/kaggle/working/run_config_cj.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    r = subprocess.run([sys.executable, "train_cj.py", "--exp", exp, "--device", "cpu", "--smoke"],
                       capture_output=True, text=True, cwd="/kaggle/working")
    lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
    print(f"===== {exp}: exit code {r.returncode}")
    print("\n".join(lines[-2:] if r.returncode == 0 else lines[-25:]))

## N10. Train the session's experiments on the GPUs + what each reached on its own

In [ ]:
# N10 — train SESSION_EXPERIMENTS: one experiment per GPU at a time; a distillation student waits for its teacher,
# a train_on_all run for its epochs_from run. Progress every 5 minutes.
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_cj.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending, running, finished = list(SESSION_EXPERIMENTS), {}, {}

def ready(exp):
    e = C["experiments"][exp]
    waits = [x for x in (e.get("teacher"), e.get("epochs_from")) if x]
    return all(t not in pending and t not in [r for r, _ in running.values()] for t in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][1].poll() is not None:
            exp, p = running.pop(dev); finished[exp] = p.returncode
            print(time.strftime("%H:%M"), f"| {exp} finished on {dev} with exit code {p.returncode}", flush=True)
        if dev not in running:
            nxt = next((e for e in pending if ready(e)), None)
            if nxt:
                pending.remove(nxt)
                out = open(f"/kaggle/working/{nxt}.out", "w")
                running[dev] = (nxt, subprocess.Popen([sys.executable, "train_cj.py", "--exp", nxt, "--device", dev],
                                                      stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt} on {dev}", flush=True)
    if time.time() - last_print > 300 and running:
        for dev, (exp, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{exp}.out").read().splitlines() if l.startswith(exp)]
            print(time.strftime("%H:%M"), "|", (lines[-1] if lines else f"{exp}: starting...")[:230], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", finished or "nothing trained this session")
for exp, code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {exp} FAILED — last lines:\n" + open(f"/kaggle/working/{exp}.out").read()[-3000:])

In [ ]:
# N10b — what each run reached on its own (best checkpoint, its own section + pooling, threshold 0.5)
import json, glob, os
import cj_common as L
rows = [json.load(open(f"{f}/summary.json")) for e, f in L.find_runs("summary.json").items()
        if os.path.exists(f"{f}/test_scores.pkl")]          # this session's runs + the finished earlier ones
print(f"{'run':22s} {'arch':9s} {'method':8s} {'train%':>7s} {'dev F1':>7s} {'F1 c0':>6s} {'F1 c1':>6s} {'epochs':>7s} {'min':>5s} {'GPU GB':>6s}")
for s in sorted(rows, key=lambda s: -(s.get("dev_macro_f1") or -1)):
    f = lambda k: f"{s[k]:.3f}" if s.get(k) is not None else "  n/a"
    print(f"{s['exp']:22s} {s['arch']:9s} {s['method']:8s} {100 * s['params_trainable'] / s['params_total']:7.2f} "
          f"{f('dev_macro_f1'):>7s} {f('dev_f1_0'):>6s} {f('dev_f1_1'):>6s} {s['epochs_done']:7.2f} {s['train_minutes']:5.0f} {str(s['peak_gpu_gb']):>6s}")

## N11. The lab — prediction sets, zips, index.csv, score chart

In [ ]:
# N11 — the lab (CPU, a few minutes): every finished run x SELECTS x token aggregation x THRESHOLDS; your
# PROB_ENSEMBLES; greedy ensembles. Writes zips, index.csv and the score chart.
import importlib, cj_common as L, cj_lab
importlib.reload(L); importlib.reload(cj_lab)
lab = cj_lab.Lab()
lab.load_runs(RUN_DIR_OVERRIDES)
lab.singles(SELECTS, THRESHOLDS)
lab.ensembles(PROB_ENSEMBLES, SELECTS, THRESHOLDS)
if GREEDY:
    lab.greedy(SELECTS, THRESHOLDS)
rows = lab.finish(TOP_N_ZIPS)
cj_lab.table(rows, 30)
cj_lab.chart(rows, "/kaggle/working/submissions/score_chart.png")
from IPython.display import Image, display
display(Image("/kaggle/working/submissions/score_chart.png"))

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, ids, values) and confirm the lab's dev scores with the official evaluator
# (evaluate_phase4.py) on the 5 best written prediction sets.
import zipfile, json, glob, csv, subprocess, sys
import cj_common as L
test_ids = set(L.Split("test").ids)
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip")) + sorted(glob.glob("/kaggle/working/cj_runs/*/submission_*.zip"))
bad_zips = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    ids = [r["id"] for r in pred]
    ok = (names == ["prediction.json"] and isinstance(pred, list) and set(ids) == test_ids and len(ids) == len(test_ids)
          and all(type(r["pred"]) is int and r["pred"] in (0, 1) for r in pred))
    bad_zips += not ok
    if not ok:
        print("PROBLEM", z)
print(f"{len(zips)} zips checked, {bad_zips} with problems")
V = "/kaggle/working/submissions/dev_predictions"
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv")) if r["file"].endswith(".zip") and r["dev_f1"]]
for r in rows[:5]:
    out = subprocess.run([sys.executable, "evaluate_phase4.py", "--truth", f"{V}/dev_truth.json",
                          "--prediction", f"{V}/{r['name']}.json"], capture_output=True, text=True)
    off = [l.split()[-1] for l in out.stdout.splitlines() if l.startswith("Macro-F1")] or [out.stderr.strip()[-200:]]
    print(f"{r['name'][:60]:60s} lab {r['dev_f1']} | official {off[0]}")

## N13. Error analysis of one prediction set (confusion matrix, errors by case length)

In [ ]:
# N13 — error analysis of one written prediction set (material for the report): confusion matrix, and how errors
# depend on case length.
import json, csv, numpy as np
import cj_common as L
NAME = None          # <- a name from submissions/index.csv; None = the best written one
V = "/kaggle/working/submissions/dev_predictions"
if NAME is None:
    NAME = next(r["name"] for r in csv.DictReader(open("/kaggle/working/submissions/index.csv"))
                if r["file"].endswith(".zip") and r["dev_f1"])
print("analysing:", NAME)
dev = L.Split("dev")
pred = {r["id"]: r["pred"] for r in json.load(open(f"{V}/{NAME}.json"))}
last = {i: k for k, i in enumerate(dev.ids)}            # dev.json repeats some ids: each id once (last occurrence)
ids = list(last)
g = np.array([dev.labels[last[i]] for i in ids]); p = np.array([pred[i] for i in ids]); n = dev.lengths()[[last[i] for i in ids]]
print(f"confusion (rows = gold, cols = predicted):\n          pred 0  pred 1\n  gold 0  {((g == 0) & (p == 0)).sum():6d}  {((g == 0) & (p == 1)).sum():6d}"
      f"\n  gold 1  {((g == 1) & (p == 0)).sum():6d}  {((g == 1) & (p == 1)).sum():6d}")
qs = np.quantile(n, [0, 0.25, 0.5, 0.75, 1.0])
print("\naccuracy by case length (tokens):")
for a, b in zip(qs[:-1], qs[1:]):
    m = (n >= a) & (n <= b)
    print(f"  {int(a):7d} - {int(b):7d}: {m.sum():5d} cases, accuracy {np.mean(g[m] == p[m]):.3f}, "
          f"false positives {((g == 0) & (p == 1) & m).sum()}, false negatives {((g == 1) & (p == 0) & m).sum()}")